# 猴子辨别网络

McClelland, McNaughton & O’Reilly (1995), pp. 442–444, Figure 13。

猴子学习 100 对物体的选择关系。五组记忆分别在手术前约 1、3、7、11、15 周形成；术后两周测试。我们比较海马损伤组和对照组的正确率。

## 1. 输入与网络

每个物体用 25 个二值特征表示，每位以 0.2 的概率为 1。两个物体拼成 50 位输入，经 15 个 sigmoid 隐藏单元，输出一个活动值。大于 0.5 选择第一个物体，否则选择第二个。正确物体随机指定。

In [ ]:
"""Figure 13 reconstruction: sparse object pairs, cortical SGD, decaying traces."""
from copy import deepcopy
import numpy as np
import torch
from torch import nn


class MonkeyNetwork(nn.Sequential):
    def __init__(self, seed=0):
        with torch.random.fork_rng():
            torch.manual_seed(seed)
            super().__init__(nn.Linear(50, 15), nn.Sigmoid(),
                             nn.Linear(15, 1), nn.Sigmoid())
            self.double()
            for p in self.parameters():
                nn.init.uniform_(p, -.5, .5)




## 2. 学习日程与海马痕迹

每组 20 对物体，连续 10 天每天学习两对，每对呈现 14 次，组末各再呈现一次。每次直接经历留下一个海马痕迹。痕迹强度为 exp(−0.025 × 天数)，每日重放概率为 0.1 × 强度。250 条背景关联各以 0.2 的概率每天出现，训练前先经历 100 天背景学习。

In [ ]:
def schedule():
    """Day 0 = surgery; ten acquisition days + one final exposure per set."""
    trials = {}
    for group, start in enumerate([-109, -81, -53, -25, -11]):
        for offset in range(10):
            a = 20 * group + 2 * offset
            trials[start + offset] = [a] * 14 + [a + 1] * 14
        trials[start + 10] = list(range(20 * group, 20 * group + 20))
    return trials


def trace_probability(ages, rate=.07, decay=.025):
    """Probability of retrieving at least one independent trace."""
    return 1 - np.prod(1 - rate * np.exp(-decay * np.asarray(ages)))




## 3. 训练与测试

SGD 学习率 0.03，每条关联更新一次，损失为半平方误差。下面将独立网络堆叠计算，各网络仍逐条更新。损伤组术后仅保留背景学习。对照组继续重放，测试时每条痕迹以 0.07 × 强度的概率被提取；提取失败才使用网络答案。

对照组使用海马提取的解析期望，避免额外的抽样噪声。

In [ ]:
class Cohort:
    """Independent networks stacked for speed; one trial/update per animal."""
    def __init__(self, seeds):
        models = [MonkeyNetwork(s) for s in seeds]
        self.p = nn.ParameterList([nn.Parameter(torch.stack(v))
                                  for v in zip(*(list(m.parameters()) for m in models))])
        self.rows = torch.arange(len(seeds))
        self.optimizer = torch.optim.SGD(self.p, lr=.03)

    def forward(self, x):
        w, b, v, c = self.p
        h = torch.sigmoid(torch.bmm(w, x[..., None]).squeeze(-1) + b)
        return torch.sigmoid(torch.bmm(v, h[..., None]).squeeze(-1) + c).squeeze(-1)

    def train(self, x, y, orders, capture=None):
        for step in range(max(map(len, orders), default=0)):
            active = torch.tensor([step < len(o) for o in orders])
            index = torch.tensor([o[step] if step < len(o) else 0 for o in orders])
            self.optimizer.zero_grad()
            prediction = self.forward(x[self.rows, index])
            loss = (.5 * (prediction - y[self.rows, index])**2 * active).sum()
            loss.backward()
            self.optimizer.step()
            if capture is not None:
                with torch.no_grad():
                    capture(step, index, prediction.detach(), self.forward(x[self.rows, index]))

    @torch.no_grad()
    def activities(self, x):
        return torch.stack([self.forward(x[:, i]) for i in range(100)], dim=1)

    @torch.no_grad()
    def choices(self, x):
        return self.activities(x) > .5


def simulate(subjects=200, seed=0, replay_rate=.1, progress=False):
    seeds = list(range(seed, seed + subjects))
    rngs = [np.random.default_rng(s + 10000) for s in seeds]
    x = torch.tensor(np.stack([r.binomial(1, .2, (350, 50)) for r in rngs]), dtype=torch.float64)
    y = torch.tensor(np.stack([r.binomial(1, .5, 350) for r in rngs]), dtype=torch.float64)
    cortex = Cohort(seeds)
    trials = schedule()
    trace_items, trace_days = [], []
    exposure_counts = np.zeros((subjects, 100), dtype=int)
    learning = [[[] for _ in range(5)] for _ in range(subjects)]

    def orders(day, replay=True, direct=()):
        strength = np.exp(-.025 * (day - np.array(trace_days)))
        result = []
        for row, rng in enumerate(rngs):
            background = np.flatnonzero(rng.random(250) < .2) + 100
            reinstated = np.array(trace_items, dtype=int)[rng.random(len(trace_items)) < replay_rate * strength] if replay else np.array([], dtype=int)
            # Background and replay are interleaved; direct trials retain blocked order.
            incidental = rng.permutation(np.r_[background, reinstated]).tolist()
            result.append(incidental + list(direct))
            if replay:
                exposure_counts[row] += np.bincount(reinstated, minlength=100)
        return result

    for day in range(-209, 0):
        direct = trials.get(day, [])
        daily_orders = orders(day, direct=direct)
        def capture(step, indices, before, after):
            for row, index in enumerate(indices.tolist()):
                if index in [0,20,40,60,80] and len(daily_orders[row])-len(direct) <= step < len(daily_orders[row]):
                    learning[row][index//20].append([before[row].item(),after[row].item()])
        cortex.train(x, y, daily_orders, capture if day in [-109,-81,-53,-25,-11] else None)
        trace_items.extend(direct)
        trace_days.extend([day] * len(direct))
        if progress and day % 50 == 0:
            print(f'Day {day}', flush=True)
    lesion = deepcopy(cortex)
    pre_surgery_counts = exposure_counts.copy()
    # Both groups continue experiencing their background environment after surgery.
    for day in range(0, 14):
        lesion.train(x, y, orders(day, replay=False))
        cortex.train(x, y, orders(day))
    cortical_lesion = (lesion.choices(x) == y[:, :100]).numpy().astype(float)
    cortical_control = (cortex.choices(x) == y[:, :100]).numpy().astype(float)
    retrieval = np.array([trace_probability(14 - np.array(trace_days)[np.array(trace_items) == i]) for i in range(100)])
    control = retrieval + (1 - retrieval) * cortical_control
    # Reverse groups: newest to oldest, matching the paper's horizontal axis.
    summarize = lambda z: z.reshape(subjects, 5, 20).mean(2)[:, ::-1]
    return dict(weeks=[1, 3, 7, 11, 15], subjects=subjects, seed=seed,
                lesion=summarize(cortical_lesion).tolist(),
                control=summarize(control).tolist(),
                control_cortex=summarize(cortical_control).tolist(),
                retrieval=retrieval.reshape(5, 20).mean(1)[::-1].tolist(),
                presentations=(15 + pre_surgery_counts).reshape(subjects, 5, 20).mean((0, 2))[::-1].tolist(),
                replay_rate=replay_rate,
                item_details=dict(targets=y[:, :100].tolist(),
                                  learning=learning,
                                  inputs=x[:, :100].tolist(),
                                  lesion_output=lesion.activities(x).tolist(),
                                  control_output=cortex.activities(x).tolist(),
                                  retrieval=retrieval.tolist()))


In [ ]:
torch.set_num_threads(1)
result = simulate(subjects=200, seed=0, progress=True)

## 4. 结果

前两图为论文 Figure 13 的近似读数，第三图为本次独立实现（均值 ± 模拟被试标准误）。这些读数不是原始动物数据，没有用于调整本次参数。

In [ ]:
import matplotlib.pyplot as plt
def plot(result):
    weeks = result['weeks']
    fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
    # Approximate readings from Figure 13, not raw animal records.
    references = [([.62, .64, .65, .725, .675], [.79, .82, .71, .62, .70]),
                  ([.60, .65, .677, .685, .685], [.82, .795, .745, .715, .693])]
    for ax, (lesion, control), title in zip(axes, references, ['Paper: animal data (approx.)', 'Paper: simulation (approx.)']):
        ax.plot(weeks, lesion, 'o-', color='black', mfc='white', label='Lesion')
        ax.plot(weeks, control, 's--', color='gray', mfc='white', label='Control')
        ax.set_title(title)
    # Approximate SEM endpoints read from Figure 13a; retain animal uncertainty.
    axes[0].errorbar(weeks, references[0][0],
                     yerr=[[.07,.06,.06,.04,.075],[.07,.06,.06,.035,.05]],
                     fmt='none', ecolor='black', capsize=3, linewidth=1)
    axes[0].errorbar(weeks, references[0][1],
                     yerr=[[.08,.06,.09,.11,.075],[.08,.06,.09,.11,.10]],
                     fmt='none', ecolor='gray', capsize=3, linewidth=1)
    for key, marker, color in [('lesion', 'o', 'black'), ('control', 's', 'gray')]:
        values = np.array(result[key])
        axes[2].errorbar(weeks, values.mean(0), yerr=values.std(0, ddof=1) / np.sqrt(len(values)),
                         marker=marker, color=color, mfc='white', label=key.title(), capsize=3)
    axes[2].set_title(f'PyTorch: {result["subjects"]} subjects/group')
    for ax in axes:
        ax.set(xticks=weeks, ylim=(.5, 1), xlabel='Learning-to-lesion interval (weeks)')
        ax.spines[['top', 'right']].set_visible(False)
        ax.legend(frameon=False)
    axes[0].set_ylabel('Proportion correct')
    fig.tight_layout()
    return fig



plot(result);
plt.show()

## 复现边界

论文给出的结构、四个主要参数及刺激统计保持不变。未完整报告的初始化、损失、背景刺激分布、日内次序与日期边界采用明确实现约定，详见 MONKEY_NOTES.md。这里使用二值目标、均匀 ±0.5 初始化、sigmoid、无动量。手术前组训练的中心为 6.5、20.5、48.5、76.5、104.5 天。曲线差异须保留，不把重新实现称为原始代码或精确拟合。